![image.png](img/save/3.png)

- 박스
- 가우시안
- 라플라시안(상하좌우)
- 라플라시안(대각선 포함)
- 임의의 대각선 패턴 A
- 임의의 대각선 패턴 B

In [1]:
import cv2 as cv
import numpy as np
import os

In [2]:
masks = {
    "Box": np.ones((3, 3), np.float32) / 9.0,
    
    "Gaussian": np.array([[0.0030, 0.0133, 0.0219, 0.0133, 0.0030],
                          [0.0133, 0.0596, 0.0983, 0.0596, 0.0133],
                          [0.0219, 0.0983, 0.1621, 0.0983, 0.0219],
                          [0.0133, 0.0596, 0.0983, 0.0596, 0.0133],
                          [0.0030, 0.0133, 0.0219, 0.0133, 0.0030]]),

    "Laplacian_4": np.array([[ 0, -1,  0],
                             [-1,  4, -1],
                             [ 0, -1,  0]]),
    
    "Laplacian_8": np.array([[-1, -1, -1],
                             [-1,  8, -1],
                             [-1, -1, -1]]),

    "Diagonal_1": np.array([[-1,  0,  0],
                            [ 0,  0,  0],
                            [ 0,  0,  1]]),
    
    "Diagonal_2": np.array([[-1, -1,  0],
                            [-1,  0,  1],
                            [ 0,  1,  1]])
}

In [15]:
save_path = 'img/save'
if not os.path.exists(save_path):
    os.makedirs(save_path)

cap = cv.VideoCapture('img/fine.gif')

ret, first_frame = cap.read()
if not ret:
    print("파일을 읽을 수 없습니다.")
    exit()

h, w = first_frame.shape[:2]
h, w = int(h * 0.4), int(w * 0.4)
display_h, display_w = h * 2, w * 3

fourcc = cv.VideoWriter_fourcc(*'avc1') 
out = cv.VideoWriter(
    f'{save_path}/filtered_result.mp4',
    fourcc,
    20.0,
    (display_w, display_h)
)
cap.set(cv.CAP_PROP_POS_FRAMES, 0)

while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break

    img = cv.resize(frame, dsize=(0, 0), fx=0.4, fy=0.4)

    results = []
    for name, mask in masks.items():
        # float32 연산 (컬러 채널 대응)
        filtered = cv.filter2D(img.astype(np.float32), -1, mask)

        if "Laplacian" in name or "Diagonal" in name:
            res = cv.convertScaleAbs(filtered + 128) 
        else:
            res = cv.convertScaleAbs(filtered)
        
        cv.putText(res, name, (10, 20), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        results.append(res)

    top_row = np.hstack(results[:3])
    bottom_row = np.hstack(results[3:])
    full_display = np.vstack((top_row, bottom_row))

    cv.imshow('Processing and Saving...', full_display) 
    out.write(full_display)

    if cv.waitKey(30) & 0xFF == ord('q'):
        break

# 메모리 해제
cap.release()
out.release()
cv.destroyAllWindows()
print(f"저장이 완료되었습니다: {save_path}/filtered_result.mp4")

저장이 완료되었습니다: img/save/filtered_result.mp4


In [ ]:
from IPython.display import HTML


HTML("""
<video width="600" controls autoplay loop muted>
  <source src="img/save/filtered_result.mp4" type="video/mp4">
</video>
""")

![image.png](img/fine.png)

- 박스  
    불러 처리가 되었다. : 정상적인 동작이다. 

- 가우시안  
    블러 처리가 되었다. : 박스에 비해서 (사람 눈 성능 때문에) 다름점을 아주 찾을 수 없지만 
    잘 보면 박스에 비해서 덜 뭉개진 (불, 캐릭터 다리) 것을 볼 수 있다 

- 라플라시안(상하좌우)  
     눈과 컵 캐릭터의 대략적인 몸체의 윤곽을 특징으로 잡고 있다.     

- 라플라시안(대각선 포함)  
    상하좌우에 비해서 캐릭터를 확실하게 더 많이 강조하고 있다.   
    검은색을 특징으로 잘 안잡은 상하좌우 버전과 다르게 특징으로 잡고 있다.  
    그러나 배경의 윤곽선을 특징으로 잡은 것도 보인다.   

- 임의의 대각선 패턴 A  
    다른 것에 비해서 특징적인데 물체의 상단 윤곽선은 잡지 않지만   
    하단 윤곽선 선의 경우 강조되어 있다  

- 임의의 대각선 패턴 B  
    임의의 대각선 패턴 A 와 같이 상단선이 조금 더 강화는 되어 있지만 하단이 더 강조되어 있다.  
    임의의 대각선 패턴 A가 못잡은 특징을 더 잘 잡고 있다.   
